# 第三讲：随机数生成与蒙特卡洛模拟## 什么是蒙特卡洛？蒙特卡洛不是一种算法，而是一种**思想**：用大量随机试验来逼近一个确定性的答案。比如你不知道 pi 的值，但你可以往正方形里撒 10000 个点，数圆内的比例，就能逼近 pi。撒的点越多，结果越准——这就是大数定律。量化分析里无处不在：模拟股价走势、测试策略稳健性、定价衍生品……全部基于这个思想。**学习目标**- 理解「随机数其实是确定的」——种子的作用- 掌握新一代 Generator API- 熟练使用 uniform、normal、integers 三大分布- 用蒙特卡洛方法解决实际问题---

### 导入工具`matplotlib` 是 Python 最常用的画图库，这讲用它来可视化随机分布。第一行设置中文字体——否则图表里的中文会变成方块。

In [ ]:
import numpy as npimport matplotlib.pyplot as pltplt.rcParams['font.sans-serif'] = ['Heiti SC', 'SimHei', 'Arial Unicode MS']plt.rcParams['axes.unicode_minus'] = Falseprint(f"NumPy 版本: {np.__version__}")

## 3.1 Generator API —— 新一代随机数引擎### 先理解一个反直觉的概念计算机不会「真随机」。它用数学公式，给定一个**种子（seed）**，输出一串看起来随机的数字。同样的种子永远产出同样的序列——这是科学计算的**核心需求：实验必须可复现**。### 新旧对比```旧写法:  np.random.seed(42);  np.random.rand(3)     # 全局状态，多线程不安全新写法:  rng = np.random.default_rng(42);  rng.random(3)   # 独立实例，更安全```类比：旧写法是全家人共用一个电视遥控器（互相干扰），新写法是每人一个遥控器。

In [ ]:
rng = np.random.default_rng(seed=42)print("rng.random(5):", rng.random(5))rng2 = np.random.default_rng(seed=42)print("rng2.random(5):", rng2.random(5))print("完全相同?", np.array_equal(rng.random(5), rng2.random(5)))

In [ ]:
seeds = [42, 123, 2024]for s in seeds:    rng = np.random.default_rng(seed=s)    print(f"seed={s:>4} -> 前5个随机数: {rng.random(5)}")print()rng = np.random.default_rng(seed=42)print(f"seed=42  再次确认:   {rng.random(5)}")

## 3.2 uniform —— 均匀分布每个值在指定区间内「等可能」出现。```pythonrng.uniform(low=0.0, high=1.0, size=None)```| 参数 | 含义 | 默认值 ||------|------|--------|| `low` | 下界（包含） | 0.0 || `high` | 上界（不包含） | 1.0 || `size` | 输出形状 | None（返回单个值） |> ⚠️ `high` 不包含。`uniform(0, 1, 10)` 的数永远 < 1，不等于 1。

In [ ]:
rng = np.random.default_rng(seed=42)print("[0, 1):   ", rng.uniform(0, 1, 10))print("[-5, 5):  ", rng.uniform(-5, 5, 5))print("\n3x4 矩阵 [10, 20):")print(rng.uniform(10, 20, size=(3, 4)))

### 可视化：均匀分布长什么样？画 100,000 个样本的直方图。理论上每个区间的高度应该一样（因为是「均匀」的）。红线是理论概率密度 = 1.0。

In [ ]:
rng = np.random.default_rng(seed=123)samples = rng.uniform(0, 1, 100_000)fig, ax = plt.subplots(figsize=(10, 5))ax.hist(samples, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='white')ax.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='理论概率密度 = 1.0')ax.set_title('均匀分布 U(0,1) - 100,000 个样本', fontsize=14)ax.set_xlabel('值')ax.set_ylabel('概率密度')ax.legend()plt.tight_layout()plt.show()

## 3.3 normal —— 正态分布（高斯分布）自然界和金融市场里**最常见的分布**。股价的日收益率、测量误差、身高体重……都近似正态。```pythonrng.normal(loc=0.0, scale=1.0, size=None)```| 参数 | 含义 ||------|------|| `loc` | 均值 mu（钟形曲线的最高点） || `scale` | 标准差 sigma（曲线的「胖瘦」） |### 68-95-99.7 法则- 68% 的数据在 mu +/- sigma 内- 95% 在 mu +/- 2*sigma 内- 99.7% 在 mu +/- 3*sigma 内这在量化里很有用：知道日收益率的均值和标准差，就能估计极端波动的概率。

In [ ]:
rng = np.random.default_rng(seed=42)print("标准正态 N(0,1):", rng.normal(0, 1, 5))print("N(5, 2):", rng.normal(5, 2, 5))print("\n2x3 矩阵 N(100, 15):")print(rng.normal(100, 15, size=(2, 3)))

In [ ]:
rng = np.random.default_rng(seed=42)samples = rng.normal(loc=0, scale=1, size=100_000)fig, ax = plt.subplots(figsize=(10, 5))ax.hist(samples, bins=80, density=True, alpha=0.7, color='steelblue', edgecolor='white')x = np.linspace(-4, 4, 200)pdf = np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)ax.plot(x, pdf, color='red', linewidth=2, label='理论 N(0,1) 曲线')ax.set_title('标准正态分布 N(0,1) - 100,000 个样本', fontsize=14)ax.set_xlabel('值')ax.set_ylabel('概率密度')ax.legend()plt.tight_layout()plt.show()for k, label in [(1, "68%"), (2, "95%"), (3, "99.7%")]:    pct = np.mean(np.abs(samples) < k) * 100    print(f"  |x| < {k}: {pct:.1f}%  (理论: {label})")

## 3.4 integers —— 随机整数```pythonrng.integers(low, high=None, size=None, endpoint=False)```| 参数 | 含义 ||------|------|| `low` | 下界（包含） || `high` | 上界（默认不包含） || `endpoint` | 如果 True，high 也被包含 |> ⚠️ 默认 `endpoint=False`，所以 `integers(1, 7)` 是 1~6。想包含 7 要 `endpoint=True`。

In [ ]:
rng = np.random.default_rng(seed=42)print("[0, 10): ", rng.integers(0, 10, 10))print("掷 5 次骰子:", rng.integers(1, 7, 5))print("[1, 6] 含两端:", rng.integers(1, 6, 10, endpoint=True))print("\n3x4 矩阵 [0, 100):")print(rng.integers(0, 100, size=(3, 4)))

## 3.5 其他分布速查| 方法 | 分布 | 量化用途 ||------|------|----------|| `rng.uniform(low, high)` | 均匀分布 | 等概率随机抽样 || `rng.normal(loc, scale)` | 正态分布 | 收益率建模 || `rng.integers(low, high)` | 随机整数 | 掷骰子、选股编号 || `rng.exponential(scale)` | 指数分布 | 订单到达间隔 || `rng.poisson(lam)` | 泊松分布 | 日内交易次数 || `rng.choice(array, size)` | 随机抽样 | 从候选池抽股票 |---

## 3.6 蒙特卡洛实战 ①：掷骰子 10,000 次**问题：** 公平六面骰子掷 10,000 次，各点数频率是否接近 1/6？**思路：** 用 `integers(1, 7, 10000)` 一次性生成 10000 次结果，统计频率。

In [ ]:
rng = np.random.default_rng(seed=42)n_trials = 10_000dice_rolls = rng.integers(1, 7, size=n_trials)print(f"前 20 次结果: {dice_rolls[:20]}")print(f"平均值: {dice_rolls.mean():.3f} (理论值: 3.5)")print(f"标准差: {dice_rolls.std():.3f} (理论值: 1.708)")

In [ ]:
faces, counts = np.unique(dice_rolls, return_counts=True)print("点数频率分布:")print("-" * 35)for face, count in zip(faces, counts):    freq = count / n_trials    bar = '|' * int(freq * 200)    print(f"  点数 {face}: {count:>5} 次  ({freq:.4f})  | 理论: 1/6 = {1/6:.4f}  {bar}")print("-" * 35)print(f"  总计:     {counts.sum():>5} 次")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))colors = ['#FF6B6B', '#FFA94D', '#FFD43B', '#69DB7C', '#4DABF7', '#DA77F2']axes[0].bar(faces, counts, color=colors, edgecolor='white', linewidth=1.5)axes[0].axhline(y=n_trials/6, color='black', linestyle='--', linewidth=1.5,                label=f'理论值: {n_trials/6:.0f} 次')axes[0].set_title(f'掷骰子 {n_trials:,} 次 - 频数统计', fontsize=13, fontweight='bold')axes[0].set_xlabel('点数')axes[0].set_ylabel('出现次数')axes[0].set_xticks(faces)axes[0].legend()frequencies = counts / n_trialsaxes[1].bar(faces, frequencies, color=colors, edgecolor='white', linewidth=1.5)axes[1].axhline(y=1/6, color='black', linestyle='--', linewidth=1.5,                label=f'理论概率: 1/6 = {1/6:.3f}')axes[1].set_title(f'掷骰子 {n_trials:,} 次 - 频率分布', fontsize=13, fontweight='bold')axes[1].set_xlabel('点数')axes[1].set_ylabel('频率')axes[1].set_xticks(faces)axes[1].set_ylim(0, 0.25)axes[1].legend()plt.tight_layout()plt.show()max_deviation = np.max(np.abs(frequencies - 1/6))print(f"\n最大偏差: {max_deviation:.4f} ({max_deviation / (1/6) * 100:.1f}%)")

### 收敛性验证：大数定律随着试验次数增加，频率越来越接近理论值 1/6。这就是大数定律：**样本越多，统计量越稳定。**观察下面的图：前几百次波动很大，越往后越稳定在 0.1667 附近。

In [ ]:
rng = np.random.default_rng(seed=123)n_max = 50_000rolls = rng.integers(1, 7, size=n_max)cumulative_freq = np.zeros((n_max, 6))for i in range(6):    cumulative_freq[:, i] = np.cumsum(rolls == (i + 1)) / np.arange(1, n_max + 1)fig, ax = plt.subplots(figsize=(12, 5))for i in range(6):    ax.plot(range(1, n_max + 1), cumulative_freq[:, i],            linewidth=0.8, alpha=0.8, label=f'点数 {i+1}')ax.axhline(y=1/6, color='black', linestyle='--', linewidth=1.5, label='理论值 1/6')ax.set_xlabel('试验次数')ax.set_ylabel('累计频率')ax.set_title('频率收敛曲线 - 大数定律的直观展示', fontsize=13, fontweight='bold')ax.legend(loc='center right', ncol=2)ax.set_xlim(0, n_max)ax.set_ylim(0.10, 0.22)plt.tight_layout()plt.show()

## 3.7 蒙特卡洛实战 ②：估算 pi**核心思想：** 在 2x2 的正方形内随机撒点，数有多少落在内切圆内。```正方形面积 = 2 x 2 = 4内切圆面积 = pi x 1^2 = pi圆内点数 / 总点数 = 圆面积 / 正方形面积 = pi / 4所以：pi = 4 x (圆内点数 / 总点数)```这就是蒙特卡洛的精髓：**用一个可以数出来的比例，去推算一个算不出来的数。**

In [ ]:
rng = np.random.default_rng(seed=42)n_points = 10_000x = rng.uniform(-1, 1, n_points)y = rng.uniform(-1, 1, n_points)distances = np.sqrt(x**2 + y**2)inside = distances <= 1n_inside = np.sum(inside)pi_estimate = 4 * n_inside / n_pointsprint(f"总撒点数:   {n_points:,}")print(f"圆内点数:   {n_inside:,}")print(f"圆内比例:   {n_inside / n_points:.4f}")print(f"pi 估计值:   {pi_estimate:.6f}")print(f"pi 真实值:   {np.pi:.6f}")print(f"相对误差:   {abs(pi_estimate - np.pi) / np.pi * 100:.4f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))n_show = min(2000, n_points)axes[0].scatter(x[:n_show][inside[:n_show]], y[:n_show][inside[:n_show]],               c='steelblue', s=2, alpha=0.6, label='圆内点')axes[0].scatter(x[:n_show][~inside[:n_show]], y[:n_show][~inside[:n_show]],               c='salmon', s=2, alpha=0.6, label='圆外点')theta = np.linspace(0, 2*np.pi, 200)axes[0].plot(np.cos(theta), np.sin(theta), 'black', linewidth=2, label='单位圆')rect = plt.Rectangle((-1, -1), 2, 2, fill=False, edgecolor='black', linewidth=2, linestyle='--')axes[0].add_patch(rect)axes[0].set_aspect('equal')axes[0].set_title(f'蒙特卡洛撒点 (显示前 {n_show} 个)', fontsize=13, fontweight='bold')axes[0].set_xlim(-1.1, 1.1)axes[0].set_ylim(-1.1, 1.1)axes[0].legend(loc='upper right', markerscale=3)cumulative_inside = np.cumsum(inside)cumulative_pi = 4 * cumulative_inside / np.arange(1, n_points + 1)axes[1].plot(range(1, n_points + 1), cumulative_pi, linewidth=0.8, color='steelblue')axes[1].axhline(y=np.pi, color='red', linestyle='--', linewidth=1.5,                label=f'pi 真实值 = {np.pi:.6f}')axes[1].fill_between(range(1, n_points + 1), cumulative_pi, np.pi, alpha=0.15, color='red')axes[1].set_xlabel('撒点数量')axes[1].set_ylabel('pi 估计值')axes[1].set_title(f'pi 估计值的收敛过程 (最终: {pi_estimate:.4f})', fontsize=13, fontweight='bold')axes[1].legend()plt.tight_layout()plt.show()

### 不同样本量下的精度蒙特卡洛的收敛速度是 O(1/sqrt(n))：**想多一位精度，需要 100 倍的样本量。**

In [ ]:
rng = np.random.default_rng(seed=2024)sample_sizes = [100, 500, 1000, 5000, 10_000, 50_000, 100_000, 1_000_000]print(f"{'样本量':>12}  {'pi 估计值':>12}  {'误差':>12}  {'误差率':>10}")print("-" * 55)for n in sample_sizes:    x = rng.uniform(-1, 1, n)    y = rng.uniform(-1, 1, n)    pi_est = 4 * np.sum(x**2 + y**2 <= 1) / n    error = abs(pi_est - np.pi)    error_pct = error / np.pi * 100    print(f"{n:>12,}  {pi_est:>12.6f}  {error:>12.6f}  {error_pct:>9.4f}%")

## 3.8 小结 & 自检清单| 技能 | ✓ ||------|---|| 理解「随机数其实是确定的」——种子的作用 | ☐ || 会用 `default_rng(seed)` 创建可复现的随机数 | ☐ || `uniform`：均匀分布，high 不包含 | ☐ || `normal`：正态分布，理解 loc/scale | ☐ || `integers`：随机整数，注意 endpoint | ☐ || 蒙特卡洛思想：大量试验 → 逼近答案 | ☐ || 撒点法估算 pi，理解收敛速度 O(1/sqrt(n)) | ☐ |### 关键公式```蒙特卡洛 pi = 4 x (圆内点数 / 总点数)正态分布：68% 在 mu+/-sigma，95% 在 mu+/-2*sigma，99.7% 在 mu+/-3*sigma```### 📝 笔记区新建 Markdown Cell，写下：- 用你自己的话解释蒙特卡洛方法- 为什么同样的种子会产生相同的随机序列？（这对你有什么好处？）---**下一讲：** Pandas 时间序列分析——用真实金融数据实战。